# Visite guidée de `moelab`

Ce notebook charge OLMoE avec le moteur du projet, génère une réponse, puis ouvre
le capot : ce qu'il y a dans chaque niveau de mémoire, et les décisions HOBBIT
pour un token.

Prérequis : le store est construit (`GUIDE.md`, étape 3) et, idéalement, les
bandes passantes mesurées (étape 4).

In [ ]:
import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # la racine du repo

import torch
from transformers import AutoTokenizer
from moelab import load_offloaded_model, PrecisionPolicy, RuntimeConfig
from moelab.hw import load_bandwidth
from moelab.plan import plan_report

MODEL = r"D:\models\olmoe-hf"
STORE = r"C:\moe-store\olmoe"

## 1. Chargement : seulement la partie dense

Le squelette du modèle est construit sur le device `meta` (sans poids), les modules
`experts` sont remplacés, et seuls ~0,9 Gio de poids denses sont lus. Les budgets
VRAM/RAM sont calculés d'après la mémoire libre.

In [ ]:
cfg = RuntimeConfig(fast_budget_bytes=None, host_budget_bytes=None, usage_path=STORE + r"\usage.json")
bw = load_bandwidth(STORE)
if bw and bw.get("b_pcie"):
    cfg.b_pcie, cfg.b_host = bw["b_pcie"], bw["b_host"]

t0 = time.time()
model, rt = load_offloaded_model(MODEL, STORE, cfg)
rt.warm_start(preload_lo=True)
tok = AutoTokenizer.from_pretrained(MODEL)
print(f"chargé en {time.time() - t0:.1f} s")
print(rt.describe())
for line in plan_report(rt.store, rt.cfg.fast_budget_bytes or 0, rt.cfg.host_budget_bytes):
    print(line)

## 2. Une réponse, et les statistiques du système

In [ ]:
def ask(question, max_new_tokens=60):
    ids = tok.apply_chat_template([{"role": "user", "content": question}], add_generation_prompt=True,
                                  return_tensors="pt", return_dict=False).to(model.moelab_device)
    rt.reset_stats()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=False)
    dt = time.perf_counter() - t0
    print(tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True))
    s = rt.stats.as_dict()
    print(f"\n{(out.shape[1] - ids.shape[1]) / dt:.2f} tok/s")
    for k in ("fast_hit_rate", "hi_share", "cpu_share", "disk_per_token", "prefetch_recall"):
        print(f"  {k:<16} {s[k]}")
    rt.end_of_request()

ask("Explique-moi la photosynthèse en une phrase.")

## 3. Les décisions HOBBIT pour un token

On capture les poids du routeur de la couche 0 pour le dernier token, puis on
applique la politique : score d'"unimportance" (normalisé, doc 06) et précision
choisie pour chacun des 8 experts.

In [ ]:
captured = {}
h = rt.gates[0].register_forward_hook(lambda m, i, o: captured.update(w=o[1].detach().cpu(), idx=o[2].detach().cpu()))
ids = tok("Le chat mange une souris", return_tensors="pt").input_ids.to(model.moelab_device)
with torch.no_grad():
    model(ids)
h.remove()

w, idx = captured["w"][-1:], captured["idx"][-1]
pol = PrecisionPolicy("hobbit", t1=0.6, t2=0.9)
s = pol.unimportance(w)[0]
need_hi, skip = pol.decide(0, w)
for e, wi, si, hi, sk in zip(idx.tolist(), w[0].tolist(), s.tolist(), need_hi[0].tolist(), skip[0].tolist()):
    print(f"expert {e:>2}  poids {wi:.4f}  s = {si:.3f}  ->  {'sauté' if sk else ('fp16' if hi else 'int4')}")

## 4. Même question, trois politiques

`hi` = modèle d'origine, `lo` = tout en int4, `hobbit` = selon l'importance.
Regarde la part de haute précision et la vitesse (le benchmark le fait
proprement, avec des mesures de qualité).

Attention : les caches ne sont pas vidés entre les tours, donc `hobbit` profite des
experts fp16 laissés en RAM par `hi` (une version plus précise sert gratuitement une
demande int4). Le benchmark, lui, vide les caches entre les réglages.


In [ ]:
for mode in ("hi", "lo", "hobbit"):
    rt.policy = rt.cfg.policy = PrecisionPolicy(mode)
    print(f"===== {mode}")
    ask("Donne trois conseils pour bien dormir.", max_new_tokens=40)

In [ ]:
rt.end_of_request()
rt.close()